<a href="https://colab.research.google.com/github/Swas26/NLP-ESG/blob/main/data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# news scrapper + PDF plumber


In [ ]:
!pip install pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 59.3 MB/s eta 0:00:00


In [ ]:
!pip install newsapi-python

In [ ]:
from newsapi import NewsApiClient
import pandas as pd
import time

api = NewsApiClient(api_key='822e469b13934cf2adc43ce8872be994')

#all companies
companies = ['AGL', 'CBA', 'ANZ','Qantas', 'Santos', 'NBN', 'Muji', 'LVMH', 'UniQlo',
             'Woolworths Australia',   'Coles Group', 'Gucci']

all_articles = []

#finds every single article that mentioons the companies listed above
for company in companies:
    response = api.get_everything(
        q=f'{company} sustainability OR environment OR ESG OR emissions',
        language='en',
        page_size=30,
        sort_by='relevancy'
    )

    for article in response['articles']:
        all_articles.append({
            'company': company,
            'text': str(article['title']) + '. ' + str(article['description']),
            'published_at': article['publishedAt'],
            'source': article['source']['name']
        })

    print(f"{company}: {len(response['articles'])} articles fetched")
    time.sleep(1)

# save to csv
df = pd.DataFrame(all_articles)
df.dropna(subset=['text'], inplace=True)
df.to_csv('news_articles.csv', index=False)
print(f"\nDone — {len(df)} articles saved")

AGL: 18 articles fetched
CBA: 29 articles fetched
ANZ: 29 articles fetched
Qantas: 28 articles fetched
Santos: 30 articles fetched
NBN: 30 articles fetched
Muji: 11 articles fetched
LVMH: 30 articles fetched
UniQlo: 28 articles fetched
Woolworths Australia: 30 articles fetched
Coles Group: 28 articles fetched
Gucci: 30 articles fetched

Done — 321 articles saved


In [ ]:
print(df.head())
print(df['company'].value_counts())

  company                                               text  \
0     AGL  [AGL energy] Small 40GB Mobile plan $9 (for 4 ...   
1     AGL  [NSW, QLD, VIC, SA] Switch to AGL & Stay until...   
2     AGL  Cogent Biosciences (COGT): Billionaire Stan Dr...   
3     AGL  Why agilon health (AGL) Is Doing a 1-for-25 Re...   
4     AGL  $75 Bill Credit with Any New Mobile SIM Plan @...   

           published_at               source  
0  2026-04-17T05:17:37Z     Ozbargain.com.au  
1  2026-04-06T10:19:26Z     Ozbargain.com.au  
2  2026-04-08T20:32:43Z  Yahoo Entertainment  
3  2026-03-29T20:48:23Z  Yahoo Entertainment  
4  2026-03-25T09:35:07Z     Ozbargain.com.au  
company
Woolworths Australia    30
LVMH                    30
NBN                     30
Santos                  30
Gucci                   30
CBA                     29
ANZ                     29
Qantas                  28
Coles Group             28
UniQlo                  28
AGL                     18
Muji                    11
N

In [ ]:
import pdfplumber
import pandas as pd
import os

from google.colab import drive
drive.mount('/content/drive')

pdf_folder = '/content/drive/MyDrive/NLP Assessment_3/'

#matching the company yearly financial report with the names in the list as a dictionary
filename_to_company = {
    'Woolworths Australia.pdf': 'Woolworths Australia',
    'UniQlo.pdf': 'UniQlo',
    'Santos.pdf': 'Santos',
    'Qantas.pdf': 'Qantas',
    'NBN.pdf': 'NBN',
    'Muji.pdf': 'Muji',
    'LVMH.pdf': 'LVMH',
    'Gucci.pdf': 'Gucci',
    'Coles Group.pdf': 'Coles Group',
    'CBA.pdf': 'CBA',
    'ANZ.pdf': 'ANZ',
    'AGL.pdf': 'AGL',
}

all_chunks = []

for filename, company_name in filename_to_company.items():
    filepath = os.path.join(pdf_folder, filename)

    if not os.path.exists(filepath):
        print(f"MISSING: {filename} — skipping")
        continue

    full_text = ''
    with pdfplumber.open(filepath) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                full_text += text + ' '

    # chunk into ~400 word segments
    words = full_text.split()
    chunk_size = 400
    chunks_added = 0

    for i in range(0, len(words), chunk_size):
        chunk = ' '.join(words[i:i+chunk_size])
        if len(chunk.strip()) > 50:
            all_chunks.append({
                'company': company_name,
                'source_type': 'esg_report',
                'text': chunk
            })
            chunks_added += 1

    print(f"{company_name}: {len(words)} words → {chunks_added} chunks")

esg_df = pd.DataFrame(all_chunks)
esg_df.to_csv('/content/drive/MyDrive/NLP Assessment_3/esg_chunks.csv', index=False)
print(f"\nDone — {len(esg_df)} total chunks saved")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Woolworths Australia: 75574 words → 189 chunks
UniQlo: 26583 words → 67 chunks
Santos: 141190 words → 353 chunks
Qantas: 54293 words → 136 chunks
NBN: 11146 words → 28 chunks
Muji: 58365 words → 146 chunks
LVMH: 53393 words → 134 chunks
Gucci: 6814 words → 18 chunks
Coles Group: 40158 words → 101 chunks
CBA: 4104 words → 11 chunks
ANZ: 32731 words → 82 chunks
AGL: 114962 words → 288 chunks

Done — 1553 total chunks saved


In [ ]:
esg_df = pd.read_csv('/content/drive/MyDrive/NLP Assessment_3/esg_chunks.csv')
news_df = pd.read_csv('news_articles.csv')

news_df['source_type'] = 'news'
news_df = news_df[['company', 'source_type', 'text']]

combined_df = pd.concat([esg_df, news_df], ignore_index=True)
combined_df.to_csv('/content/drive/MyDrive/NLP Assessment_3/combined.csv', index=False)

print(f"ESG chunks: {len(esg_df)}")
print(f"News articles: {len(news_df)}")
print(f"Total combined: {len(combined_df)}")
print(combined_df['source_type'].value_counts())

ESG chunks: 1553
News articles: 321
Total combined: 1874
source_type
esg_report    1553
news           321
Name: count, dtype: int64
